# 🎬 Reels — Free-GPU Talking Avatar (MuseTalk)

Generate a talking-avatar reel from a **script + a short real talking-head video**, 100% free on a Colab **T4 GPU**. Driving with REAL footage (not a still photo) is what makes the mouth sharp and the head move naturally.

**Runtime → Change runtime type → GPU (T4)** before running, then run the cells top-to-bottom.

Pipeline: `script → your cloned voice (XTTS) → MuseTalk mouth-swap on your video → 9:16 reel`.

> MuseTalk's mmcv/mmpose only ship wheels for Python 3.10 + torch 2.1, but Colab is Python 3.12.
> So cell 3 installs an **isolated Miniforge Python-3.10 env** just for MuseTalk — the notebook
> kernel is left untouched (no restart needed).

In [ ]:
# 1) Check the GPU (must show a Tesla T4 / similar)
!nvidia-smi -L
import sys; print('python', sys.version.split()[0])

In [ ]:
# 2) Get this repo (tts.py, reel_utils.py, avatars/, examples/) + MuseTalk
import os
if not os.path.isdir('/content/reels'):
    !git clone https://github.com/amsinghnavdeep/reels.git /content/reels
%cd /content/reels
!git pull -q
!pip -q install edge-tts pydub pyyaml
os.makedirs('engines', exist_ok=True)
if not os.path.isdir('engines/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git engines/MuseTalk

In [ ]:
%%bash
# 3) Isolated Python-3.10 env for MuseTalk (~8-10 min, one-time).
#    torch 2.1.0+cu118 has PREBUILT mmcv/mmdet/mmpose wheels -> no slow source build.
set -e
MF=/opt/conda
if [ ! -x $MF/bin/conda ]; then
  wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
  bash /tmp/mf.sh -b -p $MF
fi
source $MF/etc/profile.d/conda.sh
conda create -y -n muse python=3.10 >/dev/null 2>&1 || true
conda activate muse
cd /content/reels/engines/MuseTalk
# MuseTalk deps EXCEPT torch/mm* (we pin those to versions that have wheels)
grep -viE '^(torch|torchvision|torchaudio|mmcv|mmdet|mmpose|mmengine)' requirements.txt > /tmp/req.txt || cp requirements.txt /tmp/req.txt
pip -q install -r /tmp/req.txt
pip -q install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118
pip -q install -U openmim
mim install mmengine
mim install 'mmcv==2.1.0'
mim install 'mmdet==3.1.0'
# chumpy (legacy mmpose dep) fails to build under pip build isolation -> install it first, no isolation
pip -q install "setuptools<70" wheel
pip -q install --no-build-isolation chumpy
mim install 'mmpose==1.1.0'
# mmpose.apis imports mmdet, whose __init__ over-strictly caps mmcv < 2.1.0; relax it (2.1.0 works fine here)
sed -i "s/mmcv_maximum_version = '2.1.0'/mmcv_maximum_version = '2.2.0'/" /opt/conda/envs/muse/lib/python3.10/site-packages/mmdet/__init__.py
export MPLBACKEND=Agg
python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"
python -c "from mmpose.apis import inference_topdown; print('mmpose import OK')"

In [ ]:
# 4) Download MuseTalk weights (~5 GB, one-time).
#    MuseTalk's download_weights.sh uses `huggingface-cli`, which the new
#    huggingface_hub REMOVED (silently downloads nothing) -> use the Python API.
%cd /content/reels/engines/MuseTalk
import os
from huggingface_hub import hf_hub_download
os.makedirs('models/face-parse-bisent', exist_ok=True)
def dl(repo, filename, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    p = hf_hub_download(repo_id=repo, filename=filename, local_dir=local_dir)
    print('  ok:', p)
for f in ['musetalk/musetalk.json', 'musetalk/pytorch_model.bin',
          'musetalkV15/musetalk.json', 'musetalkV15/unet.pth']:
    dl('TMElyralab/MuseTalk', f, 'models')
for f in ['config.json', 'diffusion_pytorch_model.bin']:
    dl('stabilityai/sd-vae-ft-mse', f, 'models/sd-vae')
for f in ['config.json', 'pytorch_model.bin', 'preprocessor_config.json']:
    dl('openai/whisper-tiny', f, 'models/whisper')
dl('yzd-v/DWPose', 'dw-ll_ucoco_384.pth', 'models/dwpose')
# Two non-HF files (these already worked via gdown/curl):
!gdown 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O models/face-parse-bisent/79999_iter.pth
!curl -sL https://download.pytorch.org/models/resnet18-5c106cde.pth -o models/face-parse-bisent/resnet18-5c106cde.pth
import glob
print('\nweights present:')
for p in sorted(glob.glob('models/**/*', recursive=True)):
    if os.path.isfile(p): print(' ', p, round(os.path.getsize(p)/1e6, 1), 'MB')

In [ ]:
# 5) Upload your driving VIDEO (real talking-head clip) + your VOICE sample, then clone your voice.
#    Realism comes from REAL footage — MuseTalk only swaps the mouth to your new audio.
%cd /content/reels
import os, subprocess
from google.colab import files
os.makedirs('output', exist_ok=True)
print('Upload your talking-head VIDEO (.mp4) ...')
DRIVE_VIDEO = next(iter(files.upload()))
print('Upload your VOICE sample (.m4a/.wav/.mp3, 8-60s) ...')
VOICE_SRC = next(iter(files.upload()))
SCRIPT = 'examples/script.txt'   # edit examples/script.txt (double-click it in the file browser) with your words

# Optional: trim to a clean talking-head window and crop out any burned captions/logo.
#   For a 360x640 short with bottom captions, e.g.:  TRIM='-ss 15 -to 19.5'  CROP='crop=360:360:0:48'
TRIM = ''
CROP = ''
# ALWAYS normalise to a safe path: spaces / # / () in the upload name break MuseTalk's
# ffmpeg frame extraction -> 0 frames read -> 'Error ... division by zero'. TRIM/CROP applied here too.
vf = f'-vf "{CROP}"' if CROP else ''
subprocess.run(f'ffmpeg -y {TRIM} -i "{DRIVE_VIDEO}" {vf} -an -r 25 -c:v libx264 -pix_fmt yuv420p output/drive_clip.mp4', shell=True, check=True)
DRIVE_VIDEO = 'output/drive_clip.mp4'
print('driving video:', DRIVE_VIDEO)

# normalise the voice sample to 16k mono wav
subprocess.run(f'ffmpeg -y -i "{VOICE_SRC}" -ar 16000 -ac 1 output/voice_ref.wav', shell=True, check=True)

# clone your voice with XTTS-v2 in an ISOLATED py3.10 env
#   (Colab's Python 3.12 kernel + numpy 2 break coqui-tts; py3.10 is the supported combo)
!source /opt/conda/etc/profile.d/conda.sh && \
  (conda env list | grep -q '/envs/tts$' || conda create -y -q -n tts python=3.10) && \
  conda activate tts && \
  export MPLBACKEND=Agg && \
  pip -q install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 && \
  pip -q install transformers==4.35.2 && \
  pip -q install coqui-tts==0.22.1 && \
  COQUI_TOS_AGREED=1 python /content/reels/tts.py --script /content/reels/examples/script.txt \
    --engine xtts --clone /content/reels/output/voice_ref.wav --lang hi \
    --output /content/reels/output/speech.wav
#   Prefer a clean non-cloned Hindi voice instead? (no isolated env needed)
#   !python tts.py --script "$SCRIPT" --engine edge --voice hi-IN-MadhurNeural --output output/speech.wav
import os; assert os.path.exists('output/speech.wav'), 'TTS failed — see the logs above'
from IPython.display import Audio; Audio(filename='output/speech.wav')

In [ ]:
# 6) Run MuseTalk lip-sync in the isolated env (your video + cloned audio -> talking video)
import yaml, os
os.chdir('/content/reels/engines/MuseTalk')
os.environ['FFMPEG_PATH'] = '/usr/bin'
os.environ['MPLBACKEND'] = 'Agg'   # Colab sets matplotlib_inline backend, invalid in the muse env
cfg = {'task_0': {'video_path': '/content/reels/' + DRIVE_VIDEO, 'audio_path': '/content/reels/output/speech.wav'}}
os.makedirs('configs/inference', exist_ok=True)
open('configs/inference/reels.yaml', 'w').write(yaml.dump(cfg))
# MuseTalk sets save_dir_full only for video input, but cleans it up unconditionally
# -> crashes on a still-image avatar AFTER the mp4 is written. Guard the cleanup.
!sed -i 's|^            shutil.rmtree(save_dir_full)|            if get_file_type(video_path) == "video": shutil.rmtree(save_dir_full)|' scripts/inference.py
!MPLBACKEND=Agg /opt/conda/envs/muse/bin/python -m scripts.inference \
  --inference_config configs/inference/reels.yaml \
  --result_dir /content/reels/output/muse \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15 --fps 25 --batch_size 4 --use_float16 \
  --parsing_mode jaw --extra_margin 10

In [ ]:
# 7) Compose a 9:16 reel + preview/download
%cd /content/reels
import glob, os
clips = sorted(glob.glob('output/muse/**/*.mp4', recursive=True), key=os.path.getmtime)
assert clips, 'No MuseTalk output found — check the previous cell logs.'
raw = clips[-1]; print('raw clip:', raw)
!python reel_utils.py --in "$raw" --out output/reel.mp4
from google.colab import files
from IPython.display import Video
files.download('output/reel.mp4')
Video('output/reel.mp4', embed=True, width=270)